## 1. Carregamento, Limpeza de Dados e Engenharia de Variáveis
Nesta etapa inicial, realizamos o carregamento da base de dados, a conversão da coluna User_Score para tipo numérico (removendo inconsistências como "tbd"), a remoção de registos com valores ausentes e a criação de novas variáveis para enriquecer a análise estatística.

In [8]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_csv('Video_Games.csv')

In [9]:
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')

df_clean = df.dropna(subset=['Critic_Score', 'User_Score', 'Year_of_Release', 'Genre', 'Publisher']).copy()

df_clean['Year_of_Release'] = df_clean['Year_of_Release'].astype(int)

df_clean['User_Score_100'] = df_clean['User_Score'] * 10

df_clean['Score_Diff'] = df_clean['Critic_Score'] - df_clean['User_Score_100']

df_clean['Sucesso_Comercial'] = np.where(df_clean['Global_Sales'] >= 1.0, 'Alto Sucesso', 'Baixo Sucesso')

df_clean.info()

<class 'pandas.DataFrame'>
Index: 7007 entries, 0 to 16926
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              7007 non-null   int64  
 1   Name               7007 non-null   str    
 2   Platform           7007 non-null   str    
 3   Year_of_Release    7007 non-null   int64  
 4   Genre              7007 non-null   str    
 5   Publisher          7007 non-null   str    
 6   NA_Sales           7007 non-null   float64
 7   EU_Sales           7007 non-null   float64
 8   JP_Sales           7007 non-null   float64
 9   Other_Sales        7007 non-null   float64
 10  Global_Sales       7007 non-null   float64
 11  Critic_Score       7007 non-null   float64
 12  Critic_Count       7007 non-null   float64
 13  User_Score         7007 non-null   float64
 14  User_Count         7007 non-null   float64
 15  Developer          7003 non-null   str    
 16  Rating             6938 non-null   str 

## 2. Análise Exploratória dos Dados e Estatísticas Descritivas
Nesta etapa, calculamos as principais estatísticas descritivas (média, mediana, desvio padrão, mínimos e máximos) para as vendas e notas, analisamos o desempenho agrupado por género e calculamos a matriz de correlação para identificar a relação estatística entre a crítica, a avaliação do público e o volume de vendas.

In [10]:
estatisticas_gerais = df_clean[['Global_Sales', 'Critic_Score', 'User_Score_100', 'Score_Diff']].describe()
print("--- ESTATÍSTICAS DESCRITIVAS GERAIS ---")
print(estatisticas_gerais)

resumo_genero = df_clean.groupby('Genre')[['Global_Sales', 'Critic_Score', 'User_Score_100']].agg({
    'Global_Sales': ['count', 'mean', 'sum'],
    'Critic_Score': 'mean',
    'User_Score_100': 'mean'
})
print("\n--- DESEMPENHO POR GÉNERO DE JOGO ---")
print(resumo_genero)

matriz_correlacao = df_clean[['Global_Sales', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Critic_Score', 'User_Score_100']].corr()
print("\n--- MATRIZ DE CORRELAÇÃO ---")
print(matriz_correlacao)

--- ESTATÍSTICAS DESCRITIVAS GERAIS ---
       Global_Sales  Critic_Score  User_Score_100   Score_Diff
count   7007.000000   7007.000000     7007.000000  7007.000000
mean       0.779822     70.313258       71.848865    -1.535607
std        1.956307     13.863550       14.394303    12.971999
min        0.010000     13.000000        5.000000   -62.000000
25%        0.110000     62.000000       65.000000    -9.000000
50%        0.300000     72.000000       75.000000    -2.000000
75%        0.740000     81.000000       82.000000     5.000000
max       82.530000     98.000000       96.000000    66.000000

--- DESEMPENHO POR GÉNERO DE JOGO ---
             Global_Sales                    Critic_Score User_Score_100
                    count      mean      sum         mean           mean
Genre                                                                   
Action               1667  0.734991  1225.23    67.880024      70.985603
Adventure             265  0.309925    82.13    66.562264     

## 3. Validação Estatística: Testes de Hipóteses e Intervalos de Confiança
Nesta etapa, aplicamos testes t de Student para validar hipóteses sobre a igualdade entre as notas da crítica e do público, assim como a diferença no volume de vendas entre géneros de jogos. Além disso, calculamos os intervalos de confiança de 95% para estimar a precisão das médias de vendas.

In [11]:
t_stat_notas, p_val_notas = stats.ttest_rel(df_clean['Critic_Score'], df_clean['User_Score_100'])
print("--- TESTE 1: CRÍTICA vs PÚBLICO (PAREADO) ---")
print("Estatística t:", t_stat_notas)
print("p-valor:", p_val_notas)

vendas_action = df_clean[df_clean['Genre'] == 'Action']['Global_Sales']
vendas_shooter = df_clean[df_clean['Genre'] == 'Shooter']['Global_Sales']

t_stat_genero, p_val_genero = stats.ttest_ind(vendas_shooter, vendas_action, equal_var=False)
print("\n--- TESTE 2: VENDAS SHOOTER vs ACTION (INDEPENDENTE) ---")
print("Estatística t:", t_stat_genero)
print("p-valor:", p_val_genero)

def intervalo_confianca_95(dados):
    n = len(dados)
    media = np.mean(dados)
    erro_padrao = stats.sem(dados)
    margem = erro_padrao * stats.t.ppf((1 + 0.95) / 2., n - 1)
    return media, media - margem, media + margem

med_act, ic_inf_act, ic_sup_act = intervalo_confianca_95(vendas_action)
med_sht, ic_inf_sht, ic_sup_sht = intervalo_confianca_95(vendas_shooter)

print("\n--- INTERVALOS DE CONFIANÇA (95%) ---")
print("Action - Média:", med_act, "| IC:", (ic_inf_act, ic_sup_act))
print("Shooter - Média:", med_sht, "| IC:", (ic_inf_sht, ic_sup_sht))

--- TESTE 1: CRÍTICA vs PÚBLICO (PAREADO) ---
Estatística t: -9.909216570827743
p-valor: 5.375095029426446e-23

--- TESTE 2: VENDAS SHOOTER vs ACTION (INDEPENDENTE) ---
Estatística t: 3.0523927513365003
p-valor: 0.0023105460698333294

--- INTERVALOS DE CONFIANÇA (95%) ---
Action - Média: 0.73499100179964 | IC: (np.float64(0.6644375146034024), np.float64(0.8055444889958777))
Shooter - Média: 0.9551463963963962 | IC: (np.float64(0.8324512279172516), np.float64(1.0778415648755408))


## 4. Modelagem Estatística: Regressão Linear Múltipla
Nesta etapa, ajustamos um modelo de Regressão Linear Múltipla (Ordinary Least Squares - OLS) para quantificar o impacto das notas da crítica, notas dos utilizadores e do volume de análises sobre o volume de vendas globais. Avaliamos o coeficiente de determinação ($R^2$), os coeficientes de regressão ($\beta$) e a significância estatística de cada variável ($p$-valor).

In [12]:
X = df_clean[['Critic_Score', 'User_Score_100', 'Critic_Count', 'User_Count']]
X = sm.add_constant(X)
y = df_clean['Global_Sales']

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:           Global_Sales   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                  0.127
Method:                 Least Squares   F-statistic:                     255.1
Date:                Mon, 12 Oct 2026   Prob (F-statistic):          6.16e-205
Time:                        14:10:40   Log-Likelihood:                -14167.
No. Observations:                7007   AIC:                         2.834e+04
Df Residuals:                    7002   BIC:                         2.838e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.9635      0.127     -7.

### 4.1. Previsão de Vendas Globais para um Novo Lançamento
Utilizamos o modelo de Regressão Linear Múltipla ajustado para prever o volume estimado de vendas globais (em milhões de unidades) para um novo jogo hipotético com nota 85 na crítica especializada, nota 80 dos utilizadores (8.0 na escala original), 60 análises de críticos e 500 avaliações de utilizadores.

In [13]:
novo_jogo = pd.DataFrame({
    'const': [1.0],
    'Critic_Score': [85.0],
    'User_Score_100': [80.0],
    'Critic_Count': [60.0],
    'User_Count': [500.0]
})

vendas_previstas = modelo.predict(novo_jogo)
print("Vendas Globais Previstas (em milhões de unidades):", vendas_previstas.iloc[0])

Vendas Globais Previstas (em milhões de unidades): 1.7868323643708108


## 5. Exportação dos Dados Tratados para o Dashboard
Nesta etapa final, exportamos a base de dados tratada e enriquecida com as novas variáveis para um ficheiro CSV, que servirá de fonte de dados para a construção do dashboard interativo.

In [14]:
df_clean.to_csv('Video_Games_Clean.csv', index=False)